# 🏓 Pickleball Court Keypoint Detection - Training

**Mô hình:** YOLOv8-Pose (Large)  
**Dataset:** pb-9bsin (v5) từ Roboflow — 2,064 ảnh với 12 keypoints  
**Mục tiêu:** Phát hiện 12 keypoints của sân pickleball  

## Sơ đồ 12 Keypoints
```
    LEFT        CENTER       RIGHT
    ┌────────────┬────────────┐
 0  │            │            │  2       ← Background Baseline
    │            1            │
    │   Service  │  Service   │
    │    Box L   │   Box R    │
 3  │            │            │  5       ← Background Kitchen Line
    ├────────────4────────────┤
    │       KITCHEN (NVZ)     │
    │                         │
    ├─────────── NET ─────────┤
    │       KITCHEN (NVZ)     │
 6  ├────────────┼────────────┤  8       ← Foreground Kitchen Line
    │            7            │
    │   Service  │  Service   │
    │    Box L   │   Box R    │
 9  │           10            │ 11       ← Foreground Baseline
    └────────────┴────────────┘
```

## 1. Setup & Cài đặt

In [ ]:
# Kiểm tra GPU
!nvidia-smi

In [ ]:
# Cài đặt thư viện
!pip install ultralytics roboflow supervision -q

In [ ]:
# Mount Google Drive để lưu kết quả
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục lưu trữ
import os
SAVE_DIR = '/content/drive/MyDrive/pickleball_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory: {SAVE_DIR}')

## 2. Tải Dataset từ Roboflow

Dataset: **pb-9bsin** (version 5)  
- ~2,064 ảnh gốc  
- 12 keypoints per image  
- Đã chia sẵn train/val/test  

> ⚠️ **Bạn cần thay `VAwW4zaxVs978t7iLszZ` bằng Roboflow API key của bạn.**  
> Lấy API key tại: https://app.roboflow.com/settings/api

In [ ]:
from roboflow import Roboflow

# ===== THAY API KEY CỦA BẠN VÀO ĐÂY =====
ROBOFLOW_API_KEY = "VAwW4zaxVs978t7iLszZ"  # <-- THAY ĐỔI
# ==========================================

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("pb-rrerm").project("pb-9bsin")
version = project.version(5)
dataset = version.download("yolov8", location="/content/court_keypoints_dataset")

print(f"\nDataset downloaded to: {dataset.location}")

## 3. Kiểm tra & Visualize Dữ liệu

In [ ]:
import os

# Kiểm tra cấu trúc thư mục
dataset_path = "/content/court_keypoints_dataset"

for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset_path, split, 'images')
    lbl_dir = os.path.join(dataset_path, split, 'labels')
    
    n_imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f'{split}: {n_imgs} images, {n_lbls} labels')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

def visualize_keypoints(image_path, label_path, keypoint_names=None):
    """Visualize keypoints from a YOLO-Pose label file."""
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    if not lines:
        return img
    
    # Parse first annotation line
    parts = lines[0].strip().split()
    # Format: class cx cy w h kp1_x kp1_y kp1_v kp2_x kp2_y kp2_v ...
    
    # Draw bounding box
    cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
    x1 = int((cx - bw/2) * w)
    y1 = int((cy - bh/2) * h)
    x2 = int((cx + bw/2) * w)
    y2 = int((cy + bh/2) * h)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    
    # Parse keypoints (starting from index 5)
    kp_data = parts[5:]
    num_kps = len(kp_data) // 3
    
    colors = plt.cm.hsv(np.linspace(0, 1, num_kps))
    
    for i in range(num_kps):
        kx = float(kp_data[i*3]) * w
        ky = float(kp_data[i*3 + 1]) * h
        kv = float(kp_data[i*3 + 2])
        
        if kv > 0 and kx > 0 and ky > 0:
            color = tuple(int(c * 255) for c in colors[i][:3])
            cv2.circle(img, (int(kx), int(ky)), 6, color, -1)
            cv2.circle(img, (int(kx), int(ky)), 7, (255, 255, 255), 1)
            label = keypoint_names[i] if keypoint_names and i < len(keypoint_names) else str(i)
            cv2.putText(img, label, (int(kx)+8, int(ky)-8),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    
    return img

# Keypoint names
KP_NAMES = [
    '0:BG-BL-L', '1:BG-BL-M', '2:BG-BL-R',
    '3:BG-KT-L', '4:BG-KT-M', '5:BG-KT-R',
    '6:FG-KT-L', '7:FG-KT-M', '8:FG-KT-R',
    '9:FG-BL-L', '10:FG-BL-M', '11:FG-BL-R',
]

# Visualize random samples
img_dir = os.path.join(dataset_path, 'train', 'images')
lbl_dir = os.path.join(dataset_path, 'train', 'labels')

images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
samples = random.sample(images, min(6, len(images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for idx, img_name in enumerate(samples):
    ax = axes[idx // 3][idx % 3]
    lbl_name = os.path.splitext(img_name)[0] + '.txt'
    
    img_path = os.path.join(img_dir, img_name)
    lbl_path = os.path.join(lbl_dir, lbl_name)
    
    if os.path.exists(lbl_path):
        vis = visualize_keypoints(img_path, lbl_path, KP_NAMES)
        ax.imshow(vis)
    else:
        img = cv2.imread(img_path)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    
    ax.set_title(img_name, fontsize=8)
    ax.axis('off')

plt.suptitle('Court Keypoints - Sample Visualization', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Cấu hình data.yaml

Tạo file `data.yaml` đúng format cho YOLOv8-Pose với 12 keypoints.

In [ ]:
import yaml

data_config = {
    'path': '/content/court_keypoints_dataset',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['court'],
    'kpt_shape': [12, 3],  # 12 keypoints, each with x, y, visibility
    # flip_idx: hoán đổi keypoints đối xứng khi flip ngang
    # 0<->2, 3<->5, 6<->8, 9<->11 (corners), 1,4,7,10 stay (center)
    'flip_idx': [2, 1, 0, 5, 4, 3, 8, 7, 6, 11, 10, 9],
}

yaml_path = '/content/court_data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f'Created: {yaml_path}')
print('---')
with open(yaml_path, 'r') as f:
    print(f.read())

## 5. Huấn luyện YOLOv8-Pose

**Cấu hình:**
- Model: `yolov8l-pose.pt` (Large — trade-off tốt giữa accuracy và speed)
- Epochs: 150 (với early stopping patience=30)
- Image size: 640 (có thể tăng lên 1280 nếu GPU T4 đủ VRAM)
- `fliplr=0.0` — TẮT flip ngang lúc đầu cho an toàn
- Mosaic giảm xuống 0.5 để giữ cấu trúc sân

> ⚠️ **Trên Colab T4 (16GB VRAM), batch=16 với imgsz=640 thường OK.**  
> Nếu gặp OOM, giảm batch xuống 8.

In [ ]:
from ultralytics import YOLO

# Tải mô hình pretrained
model = YOLO('yolov8l-pose.pt')

# Huấn luyện
results = model.train(
    data='/content/court_data.yaml',
    epochs=150,
    imgsz=640,              # Tăng lên 1280 nếu muốn chính xác hơn
    batch=16,               # Giảm xuống 8 nếu OOM
    device=0,               # GPU
    patience=30,            # Early stopping
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    fliplr=0.0,             # TẮT flip ngang (an toàn hơn)
    mosaic=0.5,             # Giảm mosaic để giữ cấu trúc sân
    close_mosaic=20,        # Tắt mosaic ở 20 epochs cuối
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=5.0,            # Xoay nhẹ
    translate=0.1,
    scale=0.3,
    perspective=0.0001,     # Biến dạng phối cảnh nhẹ
    project='/content/runs/court',
    name='train_v1',
    exist_ok=True,
)

## 6. Đánh giá Mô hình

In [ ]:
# Đánh giá trên tập validation
metrics = model.val(
    data='/content/court_data.yaml',
    split='test',
    imgsz=640,
)

print("\n" + "="*50)
print("  KẾT QUẢ ĐÁNH GIÁ")
print("="*50)
print(f"  Box mAP@50:     {metrics.box.map50:.4f}")
print(f"  Box mAP@50-95:  {metrics.box.map:.4f}")
print(f"  Pose mAP@50:    {metrics.pose.map50:.4f}")
print(f"  Pose mAP@50-95: {metrics.pose.map:.4f}")
print("="*50)

# Mục tiêu:
# - Box mAP@50 > 0.90
# - Pose mAP@50 > 0.85

In [ ]:
# Hiển thị training curves
from IPython.display import Image, display
import os

results_dir = '/content/runs/court/train_v1'

# Loss curves
if os.path.exists(f'{results_dir}/results.png'):
    display(Image(filename=f'{results_dir}/results.png', width=900))

# Confusion matrix
if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    display(Image(filename=f'{results_dir}/confusion_matrix.png', width=600))

In [ ]:
# Test trên một số ảnh mẫu
import glob

test_images = glob.glob('/content/court_keypoints_dataset/test/images/*')[:6]

best_model = YOLO(f'{results_dir}/weights/best.pt')

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for idx, img_path in enumerate(test_images):
    ax = axes[idx // 3][idx % 3]
    
    result = best_model(img_path, verbose=False)[0]
    annotated = result.plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    ax.imshow(annotated)
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')

plt.suptitle('Court Keypoint Detection - Test Results', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Lưu Mô hình về Google Drive

In [ ]:
import shutil

# Copy best weights to Google Drive
src_weights = f'{results_dir}/weights/best.pt'
dst_weights = os.path.join(SAVE_DIR, 'court_keypoint_best.pt')

shutil.copy2(src_weights, dst_weights)
print(f'✅ Best weights saved to: {dst_weights}')

# Copy last weights too
src_last = f'{results_dir}/weights/last.pt'
dst_last = os.path.join(SAVE_DIR, 'court_keypoint_last.pt')
shutil.copy2(src_last, dst_last)
print(f'✅ Last weights saved to: {dst_last}')

# Copy training results
results_dst = os.path.join(SAVE_DIR, 'court_training_results')
if os.path.exists(results_dst):
    shutil.rmtree(results_dst)
shutil.copytree(results_dir, results_dst)
print(f'✅ Training results saved to: {results_dst}')

## 8. (Tùy chọn) Tinh chỉnh — Nếu kết quả chưa đạt

Nếu Pose mAP@50 < 0.85, hãy thử:
1. Tăng `imgsz` lên 1280
2. Bật `fliplr=0.5` (nếu kiểm tra thấy flip_idx đúng)
3. Giảm `batch` và tăng `epochs`
4. Thêm dữ liệu từ nguồn khác

In [ ]:
# === UNCOMMENT NẾU CẦN TRAIN LẠI VỚI CẤU HÌNH CAO HƠN ===

# model_v2 = YOLO('yolov8l-pose.pt')
# results_v2 = model_v2.train(
#     data='/content/court_data.yaml',
#     epochs=200,
#     imgsz=1280,           # Tăng resolution
#     batch=8,              # Giảm batch vì imgsz lớn hơn
#     device=0,
#     patience=40,
#     optimizer='AdamW',
#     lr0=0.0005,           # Learning rate thấp hơn
#     lrf=0.01,
#     fliplr=0.5,           # BẬT flip ngang
#     mosaic=0.3,
#     close_mosaic=30,
#     hsv_h=0.015,
#     hsv_s=0.5,
#     hsv_v=0.3,
#     degrees=5.0,
#     translate=0.1,
#     scale=0.3,
#     perspective=0.0001,
#     project='/content/runs/court',
#     name='train_v2',
#     exist_ok=True,
# )

---
## ✅ Hoàn tất!

Sau khi train xong, copy file `court_keypoint_best.pt` từ Google Drive vào thư mục `models/` của project:
```
pickleball-analysis/
└── models/
    └── court_keypoint_best.pt
```

Rồi chạy pipeline:
```bash
python main.py --input video.mp4 \
               --court-model models/court_keypoint_best.pt \
               --ball-model models/ball_tracker_best.pt
```